In [1]:
"""
Take the Institution name as input. Use Pydantic to define the schema for
the desired output and create a custom output parser. Invoke the Chain
and Fetch Results. Extract the below Institution related details from
Wikipedia: The founder of the Institution. When it was founded. The
current branches in the institution . How many employees are working
in it. A brief 4-line summary of the institution.
"""

'\nTake the Institution name as input. Use Pydantic to define the schema for\nthe desired output and create a custom output parser. Invoke the Chain\nand Fetch Results. Extract the below Institution related details from\nWikipedia: The founder of the Institution. When it was founded. The\ncurrent branches in the institution . How many employees are working\nin it. A brief 4-line summary of the institution.\n'

In [2]:
"""
A Pydantic schema (from Pydantic) is a Python class used to define
the structure and types of data.
It automatically validates and converts input data to ensure it matches
the expected format.

# Pydantic class = Python class with built-in data validation and type checking
"""

'\nA Pydantic schema (from Pydantic) is a Python class used to define\nthe structure and types of data.\nIt automatically validates and converts input data to ensure it matches\nthe expected format.\n\n# Pydantic class = Python class with built-in data validation and type checking\n'

In [3]:
!pip install wikipedia requests beautifulsoup4 pydantic

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11785 sha256=9d0d66f88f5b0fb704c9d4468d91698e015eef55b9e558457e40875ac9b8d9ad
  Stored in directory: c:\users\prajw\appdata\local\pip\cache\wheels\79\1d\c8\b64e19423cc5a2a339450ea5d145e7c8eb3d4aa2b150cde33b
Successfully built wikipedia


In [4]:
import wikipedia
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel
from typing import Optional

In [5]:
class Institution(BaseModel):
    name: str
    founder: Optional[str] = "Not Found"
    founded: Optional[str] = "Not Found"
    headquarters: Optional[str] = "Not Found"
    branches: Optional[str] = "Not Available"
    employees: Optional[str] = "Not Found"
    summary: str



In [6]:
def fetch_info(name):
    try:
        # Get Wikipedia page
        page = wikipedia.page(name, auto_suggest=False)
        summary = wikipedia.summary(name, sentences=4)

        # Add headers
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(page.url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, "lxml")

        founder = "Not Found"
        founded = "Not Found"
        headquarters = "Not Found"
        employees = "Not Found"

        # Find the table that has the word "infobox" in its class
        # <table class="infobox vcard">
        # the class attribute exists and contains "infobox"

        infobox = soup.find("table", class_=lambda x: x and "infobox" in x)

        if infobox:
            rows = infobox.find_all("tr")
        else:
            rows = []

        print("Infobox found:", infobox is not None)
        print("Rows found:", len(rows))   # Debug

        # Extract values
        for row in rows:
            th = row.find("th")
            td = row.find("td")

            if not th or not td:
                continue

            key = th.get_text(strip=True).lower()
            value = td.get_text(separator=" ", strip=True)

            # Flexible matching
            if any(k in key for k in ["founder", "founders"]):
                founder = value

            elif any(k in key for k in ["founded", "established", "formation"]):
                founded = value

            elif any(k in key for k in ["headquarters", "location", "campus"]):
                headquarters = value

            elif any(k in key for k in ["employees", "staff", "faculty", "academic staff"]):
                employees = value

        # Branch extraction
        content = page.content.lower()

        cities = [
            "bengaluru", "delhi", "mumbai", "chennai",
            "hyderabad", "kolkata", "pune", "mysore"
        ]

        found_cities = []
        for city in cities:
            if city in content:
                found_cities.append(city.title())

        branches = ", ".join(found_cities) if found_cities else "Not Available"

        # Create object
        inst = Institution(
            name=name,
            founder=founder,
            founded=founded,
            headquarters=headquarters,
            branches=branches,
            employees=employees,
            summary=summary
        )

        return inst

    # Handle ambiguous names, missing pages, and any unexpected errors
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Disambiguation Error: {e.options[:5]}"

    except wikipedia.exceptions.PageError:
        return "No page found."

    except Exception as e:
        return f"Error: {str(e)}"

In [7]:
name = input("Enter institution name: ")
result = fetch_info(name)

print("\n--- Institution Details ---")

if isinstance(result, Institution):
    print("Name:", result.name)
    print("Founder:", result.founder)
    print("Founded:", result.founded)
    print("Headquarters:", result.headquarters)
    print("Branches:", result.branches)
    print("Employees:", result.employees)
    print("\nSummary:\n", result.summary)
else:
    print(result)

Enter institution name:  Massachusetts Institute of Technology


Infobox found: True
Rows found: 25

--- Institution Details ---
Name: Massachusetts Institute of Technology
Founder: William Barton Rogers
Founded: April 10, 1861 ; 165 years ago ( 1861-04-10 )
Headquarters: Midsize city [ 7 ] , 166 acres (67.2 ha) [ 6 ]
Branches: Not Available
Employees: 1,090 [ 4 ]

Summary:
 The Massachusetts Institute of Technology (MIT) is a private research university in Cambridge, Massachusetts, United States. Founded in 1861 to advance "useful knowledge", the university has played a significant role in the development of many areas of technology and science.
William Barton Rogers founded MIT to accelerate American industrialization through scientific knowledge. Initially funded by a federal land grant, the institute adopted a German polytechnic model emphasizing laboratory instruction in applied science and engineering, and moved from Boston's Back Bay to its current campus in Cambridge in 1916.
